# Kaggle GPU로 RAG 임베딩 만들기

청크 데이터가 담긴 Kaggle Dataset을 추가한 뒤 이 노트북을 실행하면 `/kaggle/working/rag.sqlite3`가 생성됩니다. Kaggle은 업로드한 `.7z`를 자동으로 압축 해제해 Dataset으로 마운트할 수 있습니다. 실행 전에 Notebook 옵션에서 **Accelerator를 GPU**로, **Internet을 On**으로 바꾸세요. Internet은 저장소와 BGE-M3 모델을 받는 데 필요합니다.

완료 후 **Save Version**을 누르면 Output 탭에서 `rag.sqlite3`를 다운로드할 수 있습니다. 입력은 `chunk/네이버뉴스/{id}/chunks.jsonl` 또는 `네이버뉴스/{id}/chunks.jsonl` 구조를 지원하며, `info.json`은 선택 파일입니다.

In [ ]:
from pathlib import Path
import subprocess
import sys

REPO_URL = "https://github.com/2026WAI/embedding-datas.git"
REPO_DIR = Path("/kaggle/working/embedding-datas")

if REPO_DIR.exists():
    subprocess.run(["git", "-C", str(REPO_DIR), "pull", "--ff-only"], check=True)
else:
    subprocess.run(["git", "clone", "--depth", "1", REPO_URL, str(REPO_DIR)], check=True)

subprocess.run(
    [sys.executable, "-m", "pip", "install", "-q", "-r", str(REPO_DIR / "requirements.txt")],
    check=True,
)

import torch
if not torch.cuda.is_available():
    raise RuntimeError("GPU를 찾지 못했습니다. Kaggle Notebook 옵션에서 Accelerator를 GPU로 바꾼 뒤 다시 실행하세요.")
print(f"GPU: {torch.cuda.get_device_name(0)}")

In [ ]:
# 오른쪽 패널의 Input에 추가한 Dataset의 실제 slug입니다. 다른 이름이면 이 값만 바꾸세요.
INPUT_DIR = Path("/kaggle/input/wai-test-chunk")
if not INPUT_DIR.is_dir():
    mounted = [str(path) for path in Path("/kaggle/input").iterdir()]
    raise FileNotFoundError(f"Dataset 디렉터리가 없습니다: {INPUT_DIR}\n현재 마운트: {mounted}")

# Dataset에 최상위 chunk/가 있으면 그 안을, 없으면 Dataset 루트를 사용합니다.
CHUNK_ROOT = INPUT_DIR / "chunk" if (INPUT_DIR / "chunk").is_dir() else INPUT_DIR
print(f"입력 루트: {CHUNK_ROOT}")

In [ ]:
from collections import Counter

chunk_files = sorted(CHUNK_ROOT.rglob("chunks.jsonl"))
json_files = sorted(CHUNK_ROOT.rglob("chunks.json"))
info_files = sorted(CHUNK_ROOT.rglob("info.json"))

if not chunk_files:
    if json_files:
        raise ValueError("chunks.json 파일만 발견했습니다. 현재 동기화 도구의 입력 형식은 JSON Lines인 chunks.jsonl입니다.")
    raise FileNotFoundError("chunks.jsonl을 찾지 못했습니다. 압축 안의 디렉터리 구조를 확인하세요.")

sources = Counter(path.relative_to(CHUNK_ROOT).parts[0] for path in chunk_files)
print(f"입력 루트: {CHUNK_ROOT}")
print(f"chunks.jsonl: {len(chunk_files):,}개, info.json: {len(info_files):,}개")
print("소스별 chunks.jsonl 수:")
for source, count in sources.most_common():
    print(f"  - {source}: {count:,}")
print("예시:", chunk_files[0].relative_to(CHUNK_ROOT))

In [ ]:
subprocess.run([sys.executable, "download_model.py"], cwd=REPO_DIR, check=True)

In [ ]:
# T4 16 GB 기준으로 안전한 시작값입니다. CUDA 메모리가 충분하면 64로 올릴 수 있습니다.
import os
import pty
import select

BATCH_SIZE = 32
DB_PATH = Path("/kaggle/working/rag.sqlite3")
LOG_PATH = Path("/kaggle/working/sync_embeddings.log")
command = [
    sys.executable, "-u", "sync_embeddings.py", "--rebuild",
    "--chunk-dir", str(CHUNK_ROOT),
    "--db-path", str(DB_PATH),
    "--device", "cuda",
    "--batch-size", str(BATCH_SIZE),
]

master_fd, slave_fd = pty.openpty()
process = subprocess.Popen(command, cwd=REPO_DIR, stdout=slave_fd, stderr=slave_fd, close_fds=True)
os.close(slave_fd)
output = []
try:
    while process.poll() is None:
        ready, _, _ = select.select([master_fd], [], [], 0.2)
        if ready:
            try:
                text = os.read(master_fd, 4096).decode("utf-8", errors="replace")
            except OSError:
                break
            output.append(text)
            print(text, end="", flush=True)
    while True:
        try:
            text = os.read(master_fd, 4096).decode("utf-8", errors="replace")
        except OSError:
            break
        if not text:
            break
        output.append(text)
        print(text, end="", flush=True)
finally:
    os.close(master_fd)

returncode = process.wait()
LOG_PATH.write_text("".join(output), encoding="utf-8")
if returncode:
    raise subprocess.CalledProcessError(returncode, command)
print(f"\n동기화 로그 저장 완료: {LOG_PATH}")

In [ ]:
if not DB_PATH.exists():
    raise FileNotFoundError(f"생성된 DB가 없습니다: {DB_PATH}")
sys.path.insert(0, str(REPO_DIR))
from rag_store import connect_database

connection = connect_database(DB_PATH)
try:
    chunk_count = connection.execute("SELECT COUNT(*) FROM chunks").fetchone()[0]
    vector_count = connection.execute("SELECT COUNT(*) FROM vec_chunks").fetchone()[0]
    connection.execute("PRAGMA wal_checkpoint(TRUNCATE)")
finally:
    connection.close()

if chunk_count != vector_count:
    raise RuntimeError(f"청크({chunk_count:,})와 벡터({vector_count:,}) 수가 다릅니다.")
print(f"검증 완료: 청크/벡터 {chunk_count:,}개")
print(f"DB 크기: {DB_PATH.stat().st_size / 1024**3:.2f} GiB")
print("Save Version 후 오른쪽 Output 탭에서 rag.sqlite3를 다운로드하세요.")